# ARC3 Game Inspector
Load a game, view the grid in colour, and step through actions one at a time.

**How to use:**
1. Run Cell 1 (imports + colour setup) — do this once.
2. Run Cell 2 to load a game and see its initial state.
3. Run Cell 3 to take one action and see what changed.
4. Repeat Cell 3 with different actions as needed.
5. Cell 4 resets the game back to the start.
6. Cell 5 shows a summary of all games available.

In [ ]:
# ── Cell 1: Imports and setup ────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
from arc_agi import Arcade, OperationMode
from arcengine import GameAction

# ARC3 colour palette (values 0–15)
ARC3_PALETTE = [
    '#FFFFFF',  #  0 white
    '#CCCCCC',  #  1 lt-grey
    '#999999',  #  2 md-grey
    '#666666',  #  3 dk-grey
    '#333333',  #  4 vdk-grey
    '#000000',  #  5 black
    '#E53AA3',  #  6 pink
    '#FF7BCC',  #  7 lt-pink
    '#F93C31',  #  8 red
    '#1E93FF',  #  9 blue
    '#88D8F1',  # 10 lt-blue
    '#FFDC00',  # 11 yellow
    '#FF851B',  # 12 orange
    '#921231',  # 13 maroon
    '#4FCC30',  # 14 green
    '#A356D6',  # 15 purple
]
ARC3_NAMES = [
    'white', 'lt-grey', 'md-grey', 'dk-grey', 'vdk-grey', 'black',
    'pink', 'lt-pink', 'red', 'blue', 'lt-blue', 'yellow',
    'orange', 'maroon', 'green', 'purple'
]
cmap = ListedColormap(ARC3_PALETTE)

ACTION_NAMES = {
    'reset': GameAction.RESET,
    '1': GameAction.ACTION1,  '2': GameAction.ACTION2,
    '3': GameAction.ACTION3,  '4': GameAction.ACTION4,
    '5': GameAction.ACTION5,  '6': GameAction.ACTION6,
    '7': GameAction.ACTION7,
}

def show_grid(grid, title='', ax=None, figsize=(10, 10)):
    """Render a 2D grid with the ARC3 colour palette."""
    arr = np.array(grid)
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(arr, cmap=cmap, vmin=0, vmax=15, interpolation='nearest', aspect='equal')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f'{arr.shape[1]} cols')
    ax.set_ylabel(f'{arr.shape[0]} rows')
    # Draw grid lines every cell
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which='minor', color='#888888', linewidth=0.2)
    ax.tick_params(which='minor', size=0)
    # Show unique values legend
    vals = sorted(set(arr.flatten()))
    legend_txt = '  '.join(f'{v}={ARC3_NAMES[v]}' for v in vals)
    ax.set_xlabel(f'Values: {legend_txt}', fontsize=8)
    if standalone:
        plt.tight_layout()
        plt.show()

def show_diff(grid_before, grid_after, title_before='Before', title_after='After'):
    """Show before/after side by side, with changed cells highlighted."""
    a = np.array(grid_before)
    b = np.array(grid_after)
    diff = (a != b)
    n_changed = diff.sum()

    fig, axes = plt.subplots(1, 3, figsize=(22, 8))
    show_grid(a, title=title_before, ax=axes[0])
    show_grid(b, title=title_after, ax=axes[1])

    # Diff panel: white where same, red where changed
    diff_img = np.zeros_like(a)  # 0 = white = unchanged
    diff_img[diff] = 8           # 8 = red = changed
    axes[2].imshow(diff_img, cmap=cmap, vmin=0, vmax=15, interpolation='nearest', aspect='equal')
    axes[2].set_title(f'Changed cells: {n_changed}', fontsize=11)
    axes[2].set_xticks([]); axes[2].set_yticks([])

    plt.tight_layout()
    plt.show()
    return n_changed

def print_frame_info(frame, action_taken=''):
    """Print the key state fields for a frame."""
    if action_taken:
        print(f'Action taken:      {action_taken}')
    print(f'State:             {frame.state.name}')
    print(f'Levels completed:  {frame.levels_completed} / {frame.win_levels}')
    print(f'Full reset:        {frame.full_reset}')
    print(f'Available actions: {frame.available_actions}')

# Global state — holds the active game
_env = None
_last_grid = None
_game_id = None
_arcade = Arcade(operation_mode=OperationMode.OFFLINE)

print('Setup complete. Available games:')
games = sorted(set(e.game_id.split('-')[0] for e in _arcade.available_environments))
print('  ', ', '.join(games))

In [ ]:
# ── Cell 2: Load a game ───────────────────────────────────────────────────────
# Change GAME to any of the available game IDs above.
GAME = 'ar25'

_game_id = GAME
_env = _arcade.make(GAME)
obs = _env.observation_space
_last_grid = [row.tolist() for row in obs.frame[-1]]

print(f'Loaded: {GAME}')
print(f'Win condition: complete {obs.win_levels} levels')
baseline = _env.info.baseline_actions
print(f'Baseline human actions: {len(baseline) if baseline else "unknown"}')
print(f'Frame layers: {len(obs.frame)}')
print()
print_frame_info(obs, action_taken='(initial state / RESET)')

show_grid(_last_grid, title=f'{GAME} — initial state')

In [ ]:
# ── Cell 3: Take one action ───────────────────────────────────────────────────
# Change ACTION to: 'reset', '1', '2', '3', '4', '5', '6', or '7'
ACTION = '1'

if _env is None:
    print('Run Cell 2 first to load a game.')
else:
    grid_before = _last_grid
    action = ACTION_NAMES[ACTION]
    frame = _env.step(action, data={})
    grid_after = [row.tolist() for row in frame.frame[-1]]
    _last_grid = grid_after

    print_frame_info(frame, action_taken=f'ACTION{ACTION} ({action.name})')
    print()

    n = show_diff(grid_before, grid_after,
                  title_before='Before',
                  title_after=f'After ACTION{ACTION}')

In [ ]:
# ── Cell 4: Reset the game ────────────────────────────────────────────────────
# Run this to start the current game from scratch.
if _game_id is None:
    print('Run Cell 2 first to load a game.')
else:
    _env = _arcade.make(_game_id)
    obs = _env.observation_space
    _last_grid = [row.tolist() for row in obs.frame[-1]]
    print(f'Reset {_game_id}.')
    print_frame_info(obs)
    show_grid(_last_grid, title=f'{_game_id} — after reset')

In [ ]:
# ── Cell 5: Survey all games ──────────────────────────────────────────────────
# Shows win_levels and baseline action count for every available game.
print(f'{"Game":<8}  {"Win levels":<12}  {"Baseline actions":<18}  Available action IDs')
print('-' * 65)
for env_info in sorted(_arcade.available_environments, key=lambda e: e.game_id):
    gid = env_info.game_id.split('-')[0]
    env = _arcade.make(gid)
    if env is None:
        continue
    obs = env.observation_space
    baseline = env_info.baseline_actions or []
    print(f'{gid:<8}  {obs.win_levels:<12}  {len(baseline):<18}  {obs.available_actions}')

In [ ]:
# ── Cell 6: Show current grid (without taking an action) ─────────────────────
if _last_grid is None:
    print('Run Cell 2 first.')
else:
    show_grid(_last_grid, title=f'{_game_id} — current state')

In [ ]:
# ── Cell 7: Run a sequence of actions and track level progression ─────────────
# Useful once you have a hypothesis. Edit ACTIONS to your proposed solution.
ACTIONS = ['1', '2', '3', '4', '5']  # change these

if _game_id is None:
    print('Run Cell 2 first.')
else:
    env = _arcade.make(_game_id)  # fresh start
    obs = env.observation_space
    print(f'Starting {_game_id} fresh. Win at {obs.win_levels} levels.\n')

    grids = [[row.tolist() for row in obs.frame[-1]]]
    for i, act_str in enumerate(ACTIONS):
        action = ACTION_NAMES[act_str]
        frame = env.step(action, data={})
        grid = [row.tolist() for row in frame.frame[-1]]
        grids.append(grid)
        status = '✓ level up' if frame.levels_completed > (env.observation_space.levels_completed if i > 0 else 0) else ''
        print(f'  Step {i+1}: ACTION{act_str:>1}  →  levels={frame.levels_completed}/{frame.win_levels}  state={frame.state.name}  {status}')

    print(f'\nFinal: {frame.levels_completed}/{frame.win_levels} levels completed')
    if frame.state.name == 'FINISHED':
        print('GAME WON!')